# Brain Age Prediction Training Notebook per Kaggle
Questo notebook clona la repository `SFCN` da GitHub, estrae le label, e usa la funzione di training importata da `train.py`.

In [ ]:
!rm -rf SFCN # Rimuovi se esiste già una vecchia versione
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Importa il modello e la funzione di training direttamente dalla tua repository clonata
from dp_model.model_files.sfcn import SFCN
from train import train_model

## 1. Dataset Custom e Preprocessing

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            nii_path = os.path.join(subj_dir, f"{subj_id}_FLAIR_MNI152_1mm.nii")
            
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    print(f"Saltato {subj_id}: NIfTI non trovato in {nii_path}")
                    continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                print(f"Saltato {subj_id}: JSON non trovato in {json_path}")
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                print(f"Saltato {subj_id}: Chiave 'age_scan' assente in 'participant_info'.")
                continue
            
            # Label PyTorch: da 0 a 12
            try:
                age_cat = int(age_cat_val) - 1
            except Exception as e:
                print(f"Saltato {subj_id}: Errore parsing age_scan = {age_cat_val}. Errore: {e}")
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label": age_cat
            })

        if len(self.samples) == 0:
            print(f"ATTENZIONE: Nessun campione trovato. Cerca le cartelle sub-* in: {data_dir}")

    def crop_center(self, data, out_sp):
        in_sp = data.shape
        nd = np.ndim(data)
        x_crop = int((in_sp[0] - out_sp[0]) / 2)
        y_crop = int((in_sp[1] - out_sp[1]) / 2)
        z_crop = int((in_sp[2] - out_sp[2]) / 2)
        return data[x_crop:x_crop+out_sp[0], y_crop:y_crop+out_sp[1], z_crop:z_crop+out_sp[2]]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        data = self.crop_center(data, (160, 192, 160))
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        return tensor_data, sample['label']

## 2. Avvio del Training

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"
dataset = BrainAgeDataset(KAGGLE_DATA_DIR)
print(f"Trovati {len(dataset)} campioni validi.\n")

if len(dataset) > 0:
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    # Batch size basso (es. 2-4) per evitare l'esaurimento della VRAM
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilizzo dispositivo: {device}\n")
    
    # Inizializza il modello importato dalla repo (adattato a 13 classi per la categorizzazione età)
    model = SFCN(output_dim=13).to(device)
    criterion = nn.NLLLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # Lancia l'addestramento importato da train.py
    trained_model, train_losses, val_losses = train_model(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        criterion=criterion, 
        optimizer=optimizer, 
        device=device, 
        epochs=50, 
        patience=5
    )
